Notebook to create an overview of the Data Envelopes csvs with counts and values (generated with xslt transformations). See scripts at: the github repository.
Created 2026-03-15 by Liliana Melgar

# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import csv
import re

# Widen the Jupyter Notebook container
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Set Pandas display options
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full column content
pd.set_option('display.expand_frame_repr', False)  # Disable breaking lines
pd.set_option('display.width', 0)

# Clean df.info display
import io
buf = io.StringIO()

# to add timestamp to file names
import time

# disable warning for false positive on chained assignment
pd.options.mode.chained_assignment = None  # default='warn'  

# # pathlib for reading paths to files
# from pathlib import Path
# import os.path to add paths to files
import os

# import os.path to add paths to files and psutil to measure memory use in this notebook
import os, psutil

# Set up paths

In [ ]:
# Note: these paths are adapted to the jupyter server.
# overall path
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))  # Moves up two levels to reach 'repo'
# Get the current working directory
script_dir = os.getcwd()

# Define the project root relative to the current directory
project_root = os.path.abspath(os.path.join(script_dir, "..", ".."))
# Define the data directory relative to the project root
data_directory = os.path.join(project_root, "data-envelopes", "data")
# data_directory_cleaned = os.path.join(data_directory, "original_cleaned")
data_directory_downloads = os.path.join(data_directory, "downloads")


In [ ]:
print(data_directory)

# Common functions

In [ ]:
# ##### DATA TYPE CONVERTER & FILLING EMPTY VALUES (all treated as string and removing .0)
# ## This one works for dfs that don't have columns with cells as lists, only for flat strings everywhere
# def clean_df(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy()
#     # 1) fill any missing with "null" and make everything string
#     df = df.fillna("null").astype(str)

#     # 2) remove any trailing ".0" from all string values
#     # "1637.0" -> "1637", "7.0" -> "7", leaves "10.5", "abc", "null" unchanged
#     df = df.replace(r'\.0$', '', regex=True)

#     return df


# ####### CLEANERS #######
# import pandas as pd
# import unicodedata

# # --- core normalizer used by three methods ---
# #### SINGLE / CANONICAL CLEANER (trim trealing and end white spaces, collapse consequtive white spaces
# def normalize_string(s):
#     if pd.isna(s):
#         return s
#     s = str(s)
#     s = unicodedata.normalize("NFKC", s)   # normalize unicode
#     s = s.replace('\xa0', ' ')             # non-breaking space
#     s = s.strip()                          # trim outer whitespace
#     s = ' '.join(s.split())                # collapse internal whitespace
#     return s


# ### CLEAN AND SPLIT
# # normalizes and converts a dataframe column of (separator can be chosen) strings into lists of cleaned tokens
# def clean_split_column_to_list(df, col):
#     def process_cell(value):
#         if pd.isna(value):
#             return []

#         parts = str(value).split("%")
#         result = []

#         for x in parts:
#             norm = normalize_string(x.strip())
#             if norm:
#                 result.append(norm)

#         return result

#     df[col] = df[col].apply(process_cell)
#     return df

# ####### AGGREGATION FUNCTIONS FOR GROUPBY #######
# # METHOD 1: aggregation to join and keep only unique values (no reparsing)
# def agg_join_unique(series):
#     return ";".join(
#         series
#         .dropna()
#         .astype(str)
#         .str.strip()
#     )


# VERSION 1

## Value counts v1

In [ ]:
envelopes_counts_v1 = pd.read_csv(f"{data_directory}/records-v1-countsOverview.csv", sep = ",", quotechar='"', index_col=False, encoding='utf-8',engine='python')

In [ ]:
envelopes_counts_v1.head(2)

In [ ]:
envelopes_counts_v1.info(verbose=True)

In [ ]:
## create column total which counts the number of values filled in per column
envelopes_counts_v1['total'] = envelopes_counts_v1.select_dtypes(include='int64').sum(axis=1)

In [ ]:
envelopes_counts_v1_totals = envelopes_counts_v1[['record_nr','total', 'record_id']]
envelopes_counts_v1_totals.sort_values(by='total',ascending=False)

## Values (content) v1

In [ ]:
envelopes_values_v1 = pd.read_csv(f"{data_directory}/records-v1-values-all.csv", sep = ",", quotechar='"', index_col=False, encoding='utf-8',engine='python')

In [ ]:
envelopes_values_v1.head(2)

In [ ]:
envelopes_values_v1.info(verbose=True)

In [ ]:
# convert record_nr column to string to be able to see the unique counts next
envelopes_values_v1['record_nr'] = envelopes_values_v1['record_nr'].astype(str)

In [ ]:
envelopes_values_v1.describe(include="all")

# VERSION 2

## Value counts v2

In [ ]:
envelopes_counts_v2 = pd.read_csv(f"{data_directory}/records-v2-countsOverview.csv", sep = ",", quotechar='"', index_col=False, encoding='utf-8',engine='python')

In [ ]:
envelopes_counts_v2.head(2)

In [ ]:
envelopes_counts_v2.info(verbose=True)

In [ ]:
## create column total which counts the number of values filled in per column
envelopes_counts_v2['total'] = envelopes_counts_v2.select_dtypes(include='int64').sum(axis=1)

In [ ]:
envelopes_counts_v2_totals = envelopes_counts_v2[['record_nr','total', 'record_id']]
envelopes_counts_v2_totals.sort_values(by='total',ascending=False)

## Values (content) v2

In [ ]:
envelopes_values_v2 = pd.read_csv(f"{data_directory}/records-v2-values-all.csv", sep = ",", quotechar='"', index_col=False, encoding='utf-8',engine='python')

In [ ]:
envelopes_values_v2.head(2)

In [ ]:
envelopes_values_v2.info(verbose=True)

In [ ]:
# convert record_nr column to string to be able to see the unique counts next
envelopes_values_v2['record_nr'] = envelopes_values_v2['record_nr'].astype(str)

In [ ]:
envelopes_values_v2.describe(include="all")

# COMPARE COUNTS

In [ ]:
# Simple comparison (align the differences on the columns)
diff_counts = envelopes_counts_v1_totals.compare(envelopes_counts_v2_totals)
diff_counts

In [ ]:
# more detailed comparison

# Get the id columns as a separate DataFrame
id_info = envelopes_counts_v1_totals.loc[diff_counts.index, ['record_nr', 'record_id']]

# Option 1: Reset the column levels on diff
result = pd.concat([id_info, diff_counts], axis=1)

result

# COMPARE VALUES (CONTENT)

In [ ]:
# read the csv that contains the column mappings between version 1 and version 2
column_mapping = pd.read_csv(f"{data_directory}/column-mapping-v1-to-v2.csv", sep = ",", quotechar='"', index_col=False, encoding='utf-8',engine='python')

In [ ]:
# Columns need to be renamed to be able to do the comparison. Build rename dict: v1_column → v2_column (only mapped columns)
rename_dict = dict(zip(
    column_mapping.loc[column_mapping['v1_column'].notna() & column_mapping['v2_column'].notna(), 'v1_column'],
    column_mapping.loc[column_mapping['v1_column'].notna() & column_mapping['v2_column'].notna(), 'v2_column']
))

# Rename v1 columns to v2 names
v1_renamed = envelopes_values_v1.rename(columns=rename_dict)

# Keep only columns that exist in both (drop v2-only new columns)
common_cols = [c for c in envelopes_values_v2.columns if c in v1_renamed.columns]
v1_common = v1_renamed[common_cols].reset_index(drop=True)
v2_common = envelopes_values_v2[common_cols].reset_index(drop=True)


In [ ]:
# Simple comparison (align the differences on the columns)
diff_values = v1_common.compare(v2_common, keep_shape=False, keep_equal=False)
diff_values

In [ ]:
## Export

In [ ]:
# # export nodes
# timestr = time.strftime("%Y%m%d-%H%M%S")
# name_file = 'nodes_emlo'

# # create full path
# nodes_file_path = os.path.join(data_directory_downloads, f"{name_file}_{timestr}.csv")

# # write CSV to that path
# nodes_gephi.to_csv(nodes_file_path, index=False) # if too big, use compression='gzip'